# Plot Back-slip inversion of the GNSS displacement measurements of interseismic locking (coupling) at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution



In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/locking/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
flag_savefig = True
# flag_savefig = False

# Define the inversion results path
resultpath = "rst_locking/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshname = "nicoyaCK4"   # same as CK2, but connecting the trench now

# Meshes with even top boundary at 0 depth
# meshname = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshname = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
# meshname = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
meshname = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshname

print(meshname)

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
print(trench.head())

# define the type of disp. data to use
type = "both"   # use both trench parallel and normal components
# type = "dip"    # use only trench normal components     

# # read in the lon and lat of stations
# # obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
# obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

# # note that the height is in m, Dt in years, the original displacement data is in mm/yr
# # the disp relative to the stable Caribbean plate will be used in the inversion
# # From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
# df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
#                  names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
#                         'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
#                         'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
# df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
# # Convert mm to m, needed for inversion
# cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
#               'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
# df[cols_to_convert] = df[cols_to_convert] / 1e3  # Convert mm to m

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])
# displacement noise standard deviations, in m 
error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())



In [ ]:
# flag to indicate if the slip inversion was bounded
# BOUNDED = True
BOUNDED = False

BOUND_TYPE = 'both'

# choose function type,  available choices are 'tanh'- Hyperbolic tangent (default), 'arctan' - Arctangent scaled by 2/π  
# 'sigmoid' - 2/(1+exp(-x)) - 1, and 'sqrt' - x/sqrt(1+x²)
FUNCTION_TYPE = 'tanh'
# FUNCTION_TYPE = 'sigmoid'

V_para_raw = 27.0     # raw trench-parallel of Cocos-Caribbean motion, mm/yr   
V_para_const = 11.0  # a constant value from trench parallel component unrelated to tectonic loading
V_norm = 78.5     # trench-normal of Cocos-Caribbean motion, mm/yr
V_para = V_para_raw - V_para_const  # net trench-parallel motion
V_plate = round(np.sqrt(V_norm**2 + V_para**2)) # net plate motion
azimuth_deg       = 33.5   # CW from North; N33.5E

print(f"Net plate motion : {V_plate} mm")
print(f"Azimuth : {azimuth_deg} degrees CW from North")

if BOUNDED:
    # Define slip bounds based on your problem
    s_strike_max = V_para / 1e3    # remove non-elastic motion
    s_dip_max = V_norm / 1e3
    if BOUND_TYPE == 'both':
        # strike_bounds=(0.0, s_strike_max)
        strike_bounds=(-s_strike_max, s_strike_max),
        dip_bounds=(0.0, s_dip_max)
        print("Constraints to both strike and dip ")

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region=[-87, -84, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]


In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1     # same as in coseismic case

# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# # the optimal weight combination comes from the L-curve test
# # rho_s = 1e7
# # rho_s = 1e8
# rho_s = 1e9

# if BOUNDED:
#     # gamma_val_H1 = 2e1
#     gamma_val_H1 = 5e1  # seems the best 
#     # gamma_val_H1 = 1e2
#     # gamma_val_H1 = 4e2
#     # gamma_val_H1 = 1e3
# else:
#     gamma_val_H1 = 1e3
    
# delta_val_L2 = gamma_val_H1 / rho_s 

# # preferred damping for the unconstrained inversion 
# if meshname == "nicoyaCK3":   # fault zone extended to the whole subduction zone
#     gamma_val_H1 = 2.5e3
#     delta_val_L2 = 1e-5
# elif meshname == "nicoyaCK4":   # smaller fault
#     gamma_val_H1 = 2.5e3  
#     delta_val_L2 = 2.5e-5


# newest preferred values for the dense mesh, as of 12/08/2025
rho_s = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution
gamma_val_H1 = 3e3
delta_val_L2 = gamma_val_H1 / rho_s 

gamma_val_H1_ampl = 3e3
delta_val_L2_ampl = gamma_val_H1_ampl / rho_s 

if BOUNDED:
    if FUNCTION_TYPE == 'tanh':
        inv_str = f"_lockbothbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        inv_str_ampl = f"_lockbothbd_w{w_h}{w_v}_gs{gamma_val_H1_ampl:.0e}_ds{delta_val_L2_ampl:.0e}"
    else:
        inv_str = f"_lockbothbd{FUNCTION_TYPE[:4]}_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        inv_str_ampl = f"_lockbothbd{FUNCTION_TYPE[:4]}_w{w_h}{w_v}_gs{gamma_val_H1_ampl:.0e}_ds{delta_val_L2_ampl:.0e}"
        
else:
    inv_str = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
    inv_str_ampl = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1_ampl:.0e}_ds{delta_val_L2_ampl:.0e}"
    # inv_str = f"_lockingboth_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"

print(meshname)
print(inv_str)
print(inv_str_ampl)


In [ ]:
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot

    return x_rot, y_rot

# Load the ground-truth displacement for each forward structure model
def read_obs_disp(datadir, resultpath, meshname, inv_str):
    outFileName = 'd_obs_' + meshname + inv_str + '.txt'
    print(outFileName)
    u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    # the saved obs disp are aligned with mesh, reverse rotation, back to geo
    u_obs['ux'], u_obs['uy'] = rot_xy(u_obs['ux'], u_obs['uy'], -rot) 
    # print(u_obs.head())

    return u_obs

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(df, u_obs, error_e, error_n, error_v):
    # convert from m to mm, horizontal components
    df_obs_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_obs['ux']*1000,
            "north_velocity": u_obs['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    # print(df_obs_h.head())
    # np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

    # convert from m to mm, vertical components
    df_obs_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_obs['uz']*1000,
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    
    return df_obs_h, df_obs_v

In [ ]:
# read in the observation disp.
u_obs = read_obs_disp(datadir, resultpath, meshname, inv_str)

# buil observation disp. vectors
df_obs_h, df_obs_v = build_disp_vector(df, u_obs, error_e, error_n, error_v)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
# mu_l = -0.9730  # ~25 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = 0.9730 # ~55 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # string for the contrast between slab and wedge
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the 3d model, value multiplied by 4, relative a reference
    # contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
    contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026
    if not use_uneven_mesh:
        het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"
        het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}"

    else:
        # het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
        # het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull_test1"    # for testing
        het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"    # amplify rel. to 1D 
        het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}_hull"
        het3d4_str = f"_DeShon3D_ref1D_{round(4.0)}_hull"    # amplify rel. to 1D 

        het1d_str = f"_DeShon1Dref"  # 1D depth-layered model

        # #####
        # ## for testing, if building 3D model from 1D interp rather than layer lookup, and amplifying makes a difference
        # het3d_str = f"_DeShon3D_ref_{round(2.5)}_hull_test1"    
        # het3d_ori_str = f"_DeShon3D_ref_{round(1.0)}_hull_test1"
        # #####

        CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
        if CG_mu_deg == 2:
            het3d_str = het3d_str + f"_CG{CG_mu_deg}"
            het3d_ori_str = het3d_ori_str + f"_CG{CG_mu_deg}"
            het3d4_str = het3d4_str + f"_CG{CG_mu_deg}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(sw_str)
print(het1d_str)
print(het3d_ori_str)
print(het3d_str)
print(het3d4_str)

# # define the structure model strings for the INVERSION 
# struc_str_inv_het = anomaly_str
# print("Inverse problem based on: ", struc_str_inv_het)

# struc_str_inv_hom = homo_str
# print("Inverse problem based on: ", struc_str_inv_hom)


In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m
km2m = 1e3  # conversion factor from km to m

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f/km2m

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
# loc_f = loc_f.iloc[::-1].reset_index(drop=True)
# print(loc_f[['lon', 'lat']].head())
# fault.head()
print(f'lenth of fault geometry: {len(loc_f)} rows')



In [ ]:
# Load per-vertex fault basis (mesh strike/dip unit vectors), aligned row-by-row with m_s_fault.
# Used by plot_inferred_slip_panel to overlay horizontal projections of the slip vectors.
# Generated by codes/save_fault_basis.py (or as a side-product of running the inversion script).
fault_basis = pd.read_csv(datadir + resultpath + 'fault_basis_' + meshname + '.txt',
                          sep=r'\s+', comment='#',
                          names=['x_m', 'y_m', 'z_m',
                                 'strike_x', 'strike_y', 'strike_z',
                                 'dip_x', 'dip_y', 'dip_z'])
print(f'fault_basis: {len(fault_basis)} rows; should match m_s_* and loc_f row counts.')


In [ ]:
# Load the predicted surface displacement, all in meters
def read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot) 
    
    u_res = u_pred.copy()
    u_res['ux'] = u_obs['ux'] - u_pred['ux']
    u_res['uy'] = u_obs['uy'] - u_pred['uy']
    u_res['uz'] = u_obs['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    # u_res.head()

    return u_pred, u_res

# Load the inferred slip on the fault, all in meters
def read_inferred_slip(datadir, resultpath, meshname, inv_str, struc_str_inv, type):
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    if type == 'both':
        m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['s_strike', 's_dip'])
        m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)

        cols_to_convert = ['s_strike', 's_dip', 'mag']
        m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
        print(m_s['s_strike'].min(), m_s['s_strike'].max())
        print(m_s['s_dip'].min(), m_s['s_dip'].max())
        print(m_s['mag'].max())
        print(m_s[m_s['mag'] == m_s['mag'].max()][['s_strike', 's_dip']])

    elif type == 'dip':
        m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['s_dip'])
        cols_to_convert = ['s_dip']
        m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
        m_s['mag'] = m_s['s_dip']
        print(m_s['s_dip'].min(), m_s['s_dip'].max())
        print(m_s['mag'].max())

    return m_s

# Load the inferred moment and potency of the slip
def read_pred_moment(datadir, resultpath, meshname, inv_str, struc_str_inv):
    # outFileName = 'moment_' + meshname + inv_str + struc_str_inv + '.txt'
    outFileName = 'moment_dip_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    # mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
    #                 names=['moment', 'mw', 'potency'])
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['moment', 'mw', 'potency', 'moment_dip', 'mw_dip', 'potency_dip'])
    print(mom.head())
    # print("Inferred moment (Nm): ", mom['moment'].values[0])
    # print("Inferred moment magnitude Mw: ", mom['Mw'].values[0])
    # print("Inferred potency (m^3): ", mom['potency'].values[0])

    return mom

In [ ]:
##### Load the results of the homogeneous inversion
# Load the predicted surface displacement, all in meters
u_pred_hom, u_res_hom = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, homo_str)
# Load the inferred slip on the fault, all in meters
m_s_hom = read_inferred_slip(datadir, resultpath, meshname, inv_str, homo_str, type)

##### Load the results of the slab/wedge model inversion
u_pred_sw, u_res_sw = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, sw_str)
m_s_sw = read_inferred_slip(datadir, resultpath, meshname, inv_str, sw_str, type)

##### Load the results of the depth-layered 1d model inversion
u_pred_1d, u_res_1d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het1d_str)
m_s_1d = read_inferred_slip(datadir, resultpath, meshname, inv_str, het1d_str, type)

##### Load the results of the original 3d model inversion
u_pred_3d_ori, u_res_3d_ori = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, het3d_ori_str)
m_s_3d_ori = read_inferred_slip(datadir, resultpath, meshname, inv_str, het3d_ori_str, type)

##### Load the results of the amplified 3d model inversion, x2.5
u_pred_3d, u_res_3d = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str_ampl, het3d_str)
m_s_3d = read_inferred_slip(datadir, resultpath, meshname, inv_str_ampl, het3d_str, type)

# ##### Load the results of the amplified 3d model inversion, x4
# u_pred_3d4, u_res_3d4 = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str_ampl, het3d4_str)
# m_s_3d4 = read_inferred_slip(datadir, resultpath, meshname, inv_str_ampl, het3d4_str, type)


In [ ]:
# Per-model s_strike vs s_dip statistics
# -------------------------------------
# Goal: quantify how large the strike-component of slip is relative to the
# dip-component, in two reference frames:
#
#   LOCAL frame    (per-facet strike/dip) — uses s_strike, s_dip as stored.
#   REGIONAL frame (N45°E, N45°W = mesh +x, +y axes) — re-projects slip onto
#       trench-perpendicular and trench-parallel directions, common to all
#       5 models.
#
# Sign conventions:
#   LOCAL: by the PDE convention, pure downdip back-slip motion returns
#          POSITIVE s_dip.
#   REGIONAL: for THIS mesh, dip_h points to -N45°E (updip = toward trench),
#          so clean back-slip gives NEGATIVE slip_N45E.
#          Physical signed range: slip_N45E ∈ [-78.5, 0], slip_N45W ∈ [-16, 0]
#   7 fault vertices at kinks/edges have zero basis (coherence < 0.5 in
#   compute_fault_basis_per_vertex); these are excluded from the mask.

BOUND_N45E = 78.5
BOUND_N45W = 16.0
BOUND_PCT  = (1, 99)   # percentile range for bound check (guards against single-point anomalies)

# Pre-compute valid-basis mask once (shared across all models)
_ss = fault_basis['strike_x'].values**2 + fault_basis['strike_y'].values**2
_dd = fault_basis['dip_x'].values**2    + fault_basis['dip_y'].values**2
valid_basis = (_ss > 1e-12) & (_dd > 1e-12)

def slip_stats(m_s, label, mag_pct=75):
    mag    = m_s['mag'].values
    thresh = np.percentile(mag, mag_pct)
    mask   = (mag > thresh) & valid_basis
    n_sig  = int(mask.sum())

    # ── LOCAL frame ──
    s_loc = m_s['s_strike'].abs().values[mask]
    d_loc = m_s['s_dip'].abs().values[mask]
    floor_loc = 0.01 * np.nanmedian(d_loc)
    ok_loc    = np.isfinite(s_loc) & (d_loc > floor_loc)
    ratio_loc = np.nanmedian(s_loc[ok_loc] / d_loc[ok_loc])
    rake_loc  = np.degrees(np.arctan2(m_s['s_strike'].values[mask],
                                      m_s['s_dip'].values[mask]))
    arake_loc = np.abs(rake_loc)

    # ── REGIONAL frame ──
    slip_N45E = (m_s['s_strike'].values * fault_basis['strike_x'].values
               + m_s['s_dip'].values    * fault_basis['dip_x'].values)
    slip_N45W = (m_s['s_strike'].values * fault_basis['strike_y'].values
               + m_s['s_dip'].values    * fault_basis['dip_y'].values)
    sNE = slip_N45E[mask]
    sNW = slip_N45W[mask]
    floor_reg = 0.01 * np.nanmedian(np.abs(sNE))
    ok_reg    = np.isfinite(sNE) & (np.abs(sNE) > floor_reg)
    ratio_reg = np.nanmedian(np.abs(sNW)[ok_reg] / np.abs(sNE)[ok_reg])
    rake_reg  = np.degrees(np.arctan2(sNW, -sNE))
    arake_reg = np.abs(rake_reg)

    # Robust percentile bounds (p1/p99)
    sNE_lo, sNE_hi = np.percentile(sNE, BOUND_PCT)
    sNW_lo, sNW_hi = np.percentile(sNW, BOUND_PCT)

    print(f"=== {label} ===  (top {100-mag_pct}% by |mag|, excl. 7 zero-basis, n={n_sig})")
    print(f"  LOCAL  median(|s_strike|/|s_dip|) = {ratio_loc:6.3f}   "
          f"[atan = {np.degrees(np.arctan(ratio_loc)):5.1f}°]")
    print(f"  REGIONAL median(|slip_N45W|/|slip_N45E|) = {ratio_reg:6.3f}   "
          f"[atan = {np.degrees(np.arctan(ratio_reg)):5.1f}°]")
    print(f"  Physical-bound check (p{BOUND_PCT[0]}/p{BOUND_PCT[1]} of |mag|>p{mag_pct}; should be in [-bound, 0]):")
    print(f"    slip_N45E: p{BOUND_PCT[0]}={sNE_lo:+8.2f}  p{BOUND_PCT[1]}={sNE_hi:+8.2f}  "
          f"|p{BOUND_PCT[0]}|={abs(sNE_lo):6.2f}  (bound=-{BOUND_N45E})")
    print(f"    slip_N45W: p{BOUND_PCT[0]}={sNW_lo:+8.2f}  p{BOUND_PCT[1]}={sNW_hi:+8.2f}  "
          f"|p{BOUND_PCT[0]}|={abs(sNW_lo):6.2f}  (bound=-{BOUND_N45W})")
    flags = []
    if sNE_lo < -BOUND_N45E: flags.append(f"slip_N45E p{BOUND_PCT[0]} ({sNE_lo:.2f}) < -{BOUND_N45E}  (over-locked)")
    if sNE_hi >  0:          flags.append(f"slip_N45E p{BOUND_PCT[1]} ({sNE_hi:.2f}) > 0  (wrong-sign slip)")
    if sNW_lo < -BOUND_N45W: flags.append(f"slip_N45W p{BOUND_PCT[0]} ({sNW_lo:.2f}) < -{BOUND_N45W}  (over-locked)")
    if sNW_hi >  0:          flags.append(f"slip_N45W p{BOUND_PCT[1]} ({sNW_hi:.2f}) > 0  (wrong-sign slip)")
    if flags:
        print(f"    ⚠ flags:"); [print(f"      - {f}") for f in flags]
    else:
        print(f"    ✓ both components within signed physical range [-bound, 0]")
    print()

slip_stats(m_s_hom,    "Hom (H)")
slip_stats(m_s_sw,     "Het K2L (S)")
slip_stats(m_s_1d,     "1D")
slip_stats(m_s_3d_ori, "Orig. 3D")
slip_stats(m_s_3d,     "Ampl. 3D")


In [ ]:
# difference between the heterogeneous and homogeneous model
# print((m_s_sw['s_dip']-m_s_hom['s_dip']).min(), (m_s_sw['s_dip']-m_s_hom['s_dip']).max())
print((m_s_sw['mag']-m_s_hom['mag']).min(), (m_s_sw['mag']-m_s_hom['mag']).max())

# print((m_s_1d['s_dip']-m_s_hom['s_dip']).min(), (m_s_1d['s_dip']-m_s_hom['s_dip']).max())
print((m_s_1d['mag']-m_s_hom['mag']).min(), (m_s_1d['mag']-m_s_hom['mag']).max())

# print((m_s_3d_ori['s_dip']-m_s_hom['s_dip']).min(), (m_s_3d_ori['s_dip']-m_s_hom['s_dip']).max())
print((m_s_3d_ori['mag']-m_s_hom['mag']).min(), (m_s_3d_ori['mag']-m_s_hom['mag']).max())

# print((m_s_3d['s_dip']-m_s_hom['s_dip']).min(), (m_s_3d['s_dip']-m_s_hom['s_dip']).max())
print((m_s_3d['mag']-m_s_hom['mag']).min(), (m_s_3d['mag']-m_s_hom['mag']).max())

# print((m_s_3d4['s_dip']-m_s_hom['s_dip']).min(), (m_s_3d4['s_dip']-m_s_hom['s_dip']).max())


In [ ]:
mom_rate_hom = read_pred_moment(datadir, resultpath, meshname, inv_str, homo_str)
mom_rate_sw = read_pred_moment(datadir, resultpath, meshname, inv_str, sw_str)
mom_rate_1d = read_pred_moment(datadir, resultpath, meshname, inv_str, het1d_str)
mom_rate_3d_ori = read_pred_moment(datadir, resultpath, meshname, inv_str, het3d_ori_str)
mom_rate_3d = read_pred_moment(datadir, resultpath, meshname, inv_str_ampl, het3d_str)
# mom_rate_3d4 = read_pred_moment(datadir, resultpath, meshname, inv_str_ampl, het3d4_str)

# n_yr = 2012-1950   # number of years for accumulating the interseismic strain
n_yr = 2012-1978   # number of years for accumulating the interseismic strain
print(n_yr)

mom_hom = mom_rate_hom['moment_dip']*n_yr
mom_sw = mom_rate_sw['moment_dip']*n_yr
mom_1d = mom_rate_1d['moment_dip']*n_yr
mom_3d_ori = mom_rate_3d_ori['moment_dip']*n_yr
mom_3d = mom_rate_3d['moment_dip']*n_yr
# mom_3d4 = mom_rate_3d4['moment_dip']*n_yr
_, _, M_w_hom = utp.moment2mag(mom_hom)
_, _, M_w_sw = utp.moment2mag(mom_sw)
_, _, M_w_1d = utp.moment2mag(mom_1d)
_, _, M_w_3d_ori = utp.moment2mag(mom_3d_ori)
_, _, M_w_3d = utp.moment2mag(mom_3d)
# _, _, M_w_3d4 = utp.moment2mag(mom_3d4)

print(mom_hom, M_w_hom)
print(mom_sw, M_w_sw)
print(mom_1d, M_w_1d)
print(mom_3d_ori, M_w_3d_ori)
print(mom_3d, M_w_3d)
# print(mom_3d4, M_w_3d4)


In [ ]:
# ── v2: shablack colorbars in the empty [1,0] panel ──────────────────────────────
# blackefine helpers with optional show_cbar flag (default True = backward compatible)

def plot_inferblack_slip_panel(fig, m_s, mom_rate, panel_idx, title_str, panel_label, region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    maxslip = 80
    slipstep = 8
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt], cmap=True)
    fig.text(text=f"Max. back-slip rate: {m_s[col2plt].max():.2f} mm/yr", x=region[0]+0.05, y=region[-2]+0.1,
             font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency rate: {mom_rate['potency'].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05, 
             y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)

    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

    # --- Slip vector overlay using per-vertex fault basis ---
    # slip_xyz = m_strike * strike_hat + m_dip * dip_hat (mesh-frame components).
    # Mesh frame is rotated +rot CCW from geographic; rotate (x_mesh, y_mesh) back to (east, north).
    slip_x_mesh = m_s['s_strike'].values * fault_basis['strike_x'].values \
                + m_s['s_dip'].values    * fault_basis['dip_x'].values
    slip_y_mesh = m_s['s_strike'].values * fault_basis['strike_y'].values \
                + m_s['s_dip'].values    * fault_basis['dip_y'].values
    slip_e, slip_n = rot_xy(slip_x_mesh, slip_y_mesh, -rot)

    slip_grid_spacing = 0.2   # degrees, coarser than the slip dot grid for legibility
    bm_e = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=slip_e,
                             region=region, spacing=f"{slip_grid_spacing}d")
    bm_n = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=slip_n,
                             region=region, spacing=f"{slip_grid_spacing}d")
    sign = 1    # -1
    df_slip_vec = pd.DataFrame({
        'x':              bm_e.iloc[:, 0],
        'y':              bm_e.iloc[:, 1],
        'east_velocity':  bm_e.iloc[:, 2] * sign,
        'north_velocity': bm_n.iloc[:, 2] * sign,
    })

    scale_slip = 80   # mm/yr per scale-arrow unit
    fig.velo(data=df_slip_vec, pen="0.2p,black",
             spec=f"e{0.75/scale_slip}/0",
             vector="0.075c+a45+p0.1p,black+ea+gblack+h0")

    # Legend: net plate-motion vector V_plate at azimuth_deg (CW from N).
    # Drawn with the SAME scale and arrow style as the slip vectors above so
    # users can compare slip magnitude/orientation against the plate convergence.
    az_rad = np.deg2rad(azimuth_deg)
    lgd = pd.DataFrame({
        'x': [region[0]+0.5], 'y': [region[-2]+0.35],
        'east_velocity':  [V_plate * np.sin(az_rad)],
        'north_velocity': [V_plate * np.cos(az_rad)],
    })
    fig.velo(data=lgd, pen="2p,black", spec=f"e{0.75/scale_slip}/0",
             vector="0.25c+a45+p1p,black+ea+gblack+h0")
    fig.text(text=f"V@-plate@-=",
             x=region[0]+0.3, y=region[-2]+0.65,
             font="6p,Helvetica,black", justify="MC")
    fig.text(text=f"{V_plate:.0f} mm/yr",
             x=region[0]+0.3, y=region[-2]+0.5,
             font="6p,Helvetica,black", justify="MC")

    if show_cbar:
        colorlabel = "Back-slip rate"
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            # fig.colorbar(frame=["af", "x+lBack-slip rates", "y+l(mm/yr)"], position="JBC+h+o0/0.5c")
            fig.colorbar(frame=[f"a{slipstep}f{slipstep}", f"x+l{colorlabel}", "y+l(mm/yr)"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

def plot_all_inferblack_slip(m_s_hom, m_s_sw, m_s_1d, m_s_3d, 
                             mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d, 
                             col2plt, region, flag_savefig=False):
    slipsybsz = "c0.11c"
    colormap = "viridis"
    colormap_diff = "roma"
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=4, figsize=("20c", "11.0c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.0c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        with fig.set_panel(panel=[0, 0]):
            plot_inferblack_slip_panel(fig, m_s_hom, mom_rate_hom, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferblack_slip_panel(fig, m_s_sw, mom_rate_sw, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferblack_slip_panel(fig, m_s_1d, mom_rate_1d, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferblack_slip_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)


    fig.show()
    if flag_savefig:
        figname = meshname + "_ref1D_backsliprate.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)


plot_all_inferblack_slip(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, 
                       mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori, 
                       'mag', region1, flag_savefig=flag_savefig)


In [ ]:
# define interseismic coupling between the two plates as the ratio of back normal slip rate
# (m_est vector) to local trench-normal convergence rate (m_est/V_norm.)
def interseismic_coupling(m_s, V_norm, V_para):
    m_s['locking_dip'] = m_s['s_dip'] / V_norm
    m_s['locking'] = m_s['mag']/ np.sqrt(V_norm**2 + V_para**2)
    m_s['locking1'] = m_s['mag'] / V_norm
    # m_s['locking2'] = 1-m_s['mag']/V_norm

    return m_s

m_s_hom = interseismic_coupling(m_s_hom, V_norm, V_para)
m_s_sw = interseismic_coupling(m_s_sw, V_norm, V_para)
m_s_1d = interseismic_coupling(m_s_1d, V_norm, V_para)
m_s_3d_ori = interseismic_coupling(m_s_3d_ori, V_norm, V_para)
m_s_3d = interseismic_coupling(m_s_3d, V_norm, V_para)
# m_s_3d4 = interseismic_coupling(m_s_3d4, V_norm, V_para)

#Do we need to scale it so that the max. is 1?
normflag = False
# normflag = True
if normflag:
    m_s_hom['locking'] = m_s_hom['locking']/m_s_hom['locking'].max()
    m_s_sw['locking'] = m_s_sw['locking']/m_s_sw['locking'].max()
    m_s_1d['locking'] = m_s_1d['locking']/m_s_1d['locking'].max()
    m_s_3d_ori['locking'] = m_s_3d_ori['locking']/m_s_3d_ori['locking'].max()
    m_s_3d['locking'] = m_s_3d['locking']/m_s_3d['locking'].max()
    # m_s_3d4['locking'] = m_s_3d4['locking']/m_s_3d4['locking'].max()

print(m_s_hom['locking'].min(), m_s_hom['locking'].max())
print(m_s_sw['locking'].min(), m_s_sw['locking'].max())
print(m_s_1d['locking'].min(), m_s_1d['locking'].max())
print(m_s_3d_ori['locking'].min(), m_s_3d_ori['locking'].max())
print(m_s_3d['locking'].min(), m_s_3d['locking'].max())
# print(m_s_3d4['locking'].min(), m_s_3d4['locking'].max())

print((m_s_sw['locking']-m_s_hom['locking']).min(), (m_s_sw['locking']-m_s_hom['locking']).max())
print((m_s_1d['locking']-m_s_hom['locking']).min(), (m_s_1d['locking']-m_s_hom['locking']).max())
print((m_s_3d_ori['locking']-m_s_hom['locking']).min(), (m_s_3d_ori['locking']-m_s_hom['locking']).max())
print((m_s_3d['locking']-m_s_hom['locking']).min(), (m_s_3d['locking']-m_s_hom['locking']).max())
# print((m_s_3d4['locking']-m_s_hom['locking']).min(), (m_s_3d4['locking']-m_s_hom['locking']).max())

In [ ]:
def vector_diff(m0, m1):
    m_diff = m0.copy()
    m_diff['s_dip'] = m1['s_dip']-m0['s_dip']
    m_diff['s_strike'] = m1['s_strike']-m0['s_strike']
    m_diff['mag'] = np.sqrt(m_diff['s_strike']**2 + m_diff['s_dip']**2)

    # assign the + or - sign depending on whose amplitude itself is larger
    m_diff['mag'] = m_diff['mag'] * np.sign(m1['mag'] - m0['mag'])

    return m_diff


In [ ]:
m_s_diff_swhom = vector_diff(m_s_hom, m_s_sw)
m_s_diff_1dhom = vector_diff(m_s_hom, m_s_1d)
m_s_diff_3dohom = vector_diff(m_s_hom, m_s_3d_ori)
m_s_diff_3dhom = vector_diff(m_s_hom, m_s_3d)
# m_s_diff_3d4hom = vector_diff(m_s_hom, m_s_3d4)

print(m_s_diff_swhom['mag'].min(), m_s_diff_swhom['mag'].max())
print((m_s_sw['mag']-m_s_hom['mag']).min(), (m_s_sw['mag']-m_s_hom['mag']).max())

print(m_s_diff_1dhom['mag'].min(), m_s_diff_1dhom['mag'].max())
print((m_s_1d['mag']-m_s_hom['mag']).min(), (m_s_1d['mag']-m_s_hom['mag']).max())

print(m_s_diff_3dohom['mag'].min(), m_s_diff_3dohom['mag'].max())
print((m_s_3d_ori['mag']-m_s_hom['mag']).min(), (m_s_3d_ori['mag']-m_s_hom['mag']).max())

print(m_s_diff_3dhom['mag'].min(), m_s_diff_3dhom['mag'].max())
print((m_s_3d['mag']-m_s_hom['mag']).min(), (m_s_3d['mag']-m_s_hom['mag']).max())

# print(m_s_diff_3d4hom['mag'].min(), m_s_diff_3d4hom['mag'].max())
# print((m_s_3d4['mag']-m_s_hom['mag']).min(), (m_s_3d4['mag']-m_s_hom['mag']).max())


In [ ]:
# directly use the vector difference
m_s_diff_swhom = interseismic_coupling(m_s_diff_swhom, V_norm, V_para)
m_s_diff_1dhom = interseismic_coupling(m_s_diff_1dhom, V_norm, V_para)
m_s_diff_3dohom = interseismic_coupling(m_s_diff_3dohom, V_norm, V_para)
m_s_diff_3dhom = interseismic_coupling(m_s_diff_3dhom, V_norm, V_para)
# m_s_diff_3d4hom = interseismic_coupling(m_s_diff_3d4hom, V_norm, V_para)

print(m_s_diff_swhom['locking'].min(), m_s_diff_swhom['locking'].max())
print(m_s_diff_1dhom['locking'].min(), m_s_diff_1dhom['locking'].max())
print(m_s_diff_3dohom['locking'].min(), m_s_diff_3dohom['locking'].max())
print(m_s_diff_3dhom['locking'].min(), m_s_diff_3dhom['locking'].max())
# print(m_s_diff_3d4hom['locking'].min(), m_s_diff_3d4hom['locking'].max())

In [ ]:
potency_col = 'potency'
col2plt = 'locking'
# col2plt = 'locking1'

# potency_col = 'potency_dip'
# col2plt = 'locking_dip'


In [ ]:
def plot_all_inferred_ratio(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.09c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "14c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max.: {m_s_hom[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"]) #, "y+l(mm)"

        # slip inferred from the slab/wedge model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tS/W inv."])
            # maxslip = m_s_sw[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max.: {m_s_sw[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"])

        # slip inferred from the original 3d model
        with fig.set_panel(panel=[1, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D inv."])
            # maxslip = m_s_3d_ori[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_ori[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_ori[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_ori[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max.: {m_s_3d_ori[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"])

        # slip inferred from the amplified 3d model
        with fig.set_panel(panel=[1, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tAmplified 3D inv."])
            # maxslip = m_s_3d[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max.: {m_s_3d[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"])


    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_sum_invslip.pdf")


# plot_all_inferred_ratio(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region1, flag_savefig=False)
# plot_all_inferred_ratio(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, col2plt, region, flag_savefig=False)
# plot_all_inferred_ratio(m_s_hom, m_s_sw, m_s_3d, col2plt, region, flag_savefig=False)


In [ ]:
region2 = [-89, -83, 7, 13]

In [ ]:
def plot_all_inferred_ratio2(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, m_s_1d, col2plt, region, flag_savefig=False, figname=None):

    slipsybsz = "c0.1c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "plasma"
    # colormap = "magma"
    # colormap = "inferno"
    # colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p",
                    COLOR_BACKGROUND="red", COLOR_FOREGROUND="blue",
                    )

        mu_model_strings = ["H", "S", "Orig. 3D", "Ampl. 3D", "1D"]

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[0]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # slipstep = maxslip/10
            # maxslip = 0.9
            slipstep = 0.05
            slipstep = 0.1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], reverse=True)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_hom[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_hom[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. locking: {m_s_hom[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c",) #, "y+l(mm)"
                # fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c+e0.15c",) #, "y+l(mm)"

            # Legend box bottom-left corner (adjust to your map's coordinates)
            x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
            # Box size
            width = 0.65    # degrees or projected units depending on your region
            height = 0.35
            # 1 — draw the gray box
            fig.plot(
                x=[x0, x0+width, x0+width, x0, x0],
                y=[y0, y0, y0+height, y0+height, y0],
                pen="0.5p,black",
                fill="lightgray",
                # transparency=30,
            )

            # Manual legend - more control and simpler
            # Position in your subplot coordinates
            x_legend = region[0] + 0.1  # adjust based on your region
            y_legend = region[2] + 0.45

            # Draw black line
            fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
                    pen="1p,black")
            fig.text(x=x_legend+0.15, y=y_legend+0.15, text="0.9 locking",
                    font="5p,Helvetica", justify="LM")

            # Draw white line
            fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
                    pen="1p,white")
            fig.text(x=x_legend+0.15, y=y_legend, text="0.5 locking",
                    font="5p,Helvetica", justify="LM")
            

        # Slip inferred from the SW model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[1]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_sw[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # slipstep = maxslip/10
            # slipstep = 0.05
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], reverse=True)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_sw[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_sw[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. locking: {m_s_sw[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c") #, "y+l(mm)"      
                # fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c+e0.15c",) #, "y+l(mm)"

        # Slip inferred from the 3D model
        with fig.set_panel(panel=[0, 2]):

            # m_s_3d_plt = m_s_3d
            # # m_s_3d_plt = m_s_3d4
            # mu_model_num = 3

            # m_s_3d_plt = m_s_3d_ori
            # mu_model_num = 2

            m_s_3d_plt = m_s_1d
            mu_model_num = 4

            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[mu_model_num]}"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_sw[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # slipstep = maxslip/10
            # slipstep = 0.05
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], reverse=True)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_plt[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d_plt[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s_3d_plt[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_3d_plt[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_plt[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d_plt[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"Max. locking: {m_s_3d_plt[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c+e0.15c",) #, "y+l(mm)"

    fig.show()

    if flag_savefig:
        # fig.savefig(figpath + meshname + "_locking.pdf")
        fig.savefig(figpath + figname)

figname = meshname + "_locking.pdf"
plot_all_inferred_ratio2(m_s_hom, m_s_sw, m_s_3d_ori, m_s_3d, m_s_1d, col2plt, region1, flag_savefig=False, figname=figname)



In [ ]:
def plot_slip_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # mu_model_strings = ["S - H", "Orig. 3D - H", "Ampl. 3D - H"]
        mu_model_strings = ["S - H", "1D - H", "Orig. 3D - H"]

        # # Slip inferred from the homogeneous model
        # with fig.set_panel(panel=[0, 0]):
        #     fig.basemap(region=region, projection="M?", frame=["a1f0.2"])
        #     # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
        #     #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
        #     maxslip = 1
        #     # maxslip = mtrue_s[col2plt].max()
        #     # maxslip = m_s_hom[col2plt].max()
        #     # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
        #     # slipstep = maxslip/10
        #     slipstep = 0.05
        #     pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
        #     grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
        #     # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

        #     fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
        #     fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
        #     fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

        #     fig.text(text=f"Max. locking: {m_s_hom[col2plt].max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

        #     scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
        #     mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        #     fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        #     grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
        #     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        #     # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
        #     with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #         fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c") #, "y+l(mm)"

        #     # Legend box bottom-left corner (adjust to your map's coordinates)
        #     x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
        #     # Box size
        #     width = 0.65    # degrees or projected units depending on your region
        #     height = 0.35
        #     # 1 — draw the gray box
        #     fig.plot(
        #         x=[x0, x0+width, x0+width, x0, x0],
        #         y=[y0, y0, y0+height, y0+height, y0],
        #         pen="0.5p,black",
        #         fill="lightgray",
        #         # transparency=30,
        #     )

        #     # Manual legend - more control and simpler
        #     # Position in your subplot coordinates
        #     x_legend = region[0] + 0.1  # adjust based on your region
        #     y_legend = region[2] + 0.45

        #     # Draw black line
        #     fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
        #             pen="1p,black")
        #     fig.text(x=x_legend+0.15, y=y_legend+0.15, text="0.9 locking",
        #             font="5p,Helvetica", justify="LM")

        #     # Draw white line
        #     fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
        #             pen="1p,white")
        #     fig.text(x=x_legend+0.15, y=y_legend, text="0.5 locking",
        #             font="5p,Helvetica", justify="LM")

        # difference between the SW heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            m_s = m_s_sw
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[0]}"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.02
            maxdiff = 0.14
            diffstep = maxdiff/10
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
                        
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")


        # difference between the original 1D heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            m_s = m_s_1d  
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[1]}"])
            # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.016
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
            
            # fig.text(text=f"Min. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}", x=region[0]+0.1, y=region[-2]+0.2, font="6p,Helvetica,black", justify="ML")
            # fig.text(text=f"Max. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")

        # difference between the original 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            m_s = m_s_3d_ori
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[2]}"])
            # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.016
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
            
            # fig.text(text=f"Min. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}", x=region[0]+0.1, y=region[-2]+0.2, font="6p,Helvetica,black", justify="ML")
            # fig.text(text=f"Max. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_locking_diff.pdf")

plot_slip_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, col2plt, region1, flag_savefig=False)


In [ ]:
def plot_slip_diff2(m_s_diff_swhom, m_s_diff_1dhom, m_s_diff_3dohom, 
                    m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
                    col2plt, region, difftype='amp_of_diff_vector', sort=True, flag_savefig=False):

    slipsybsz = "c0.11c"
    colormap = "lajolla"
    # colormap = "viridis"
    # colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # mu_model_strings = ["S - H", "Orig. 3D - H", "Ampl. 3D - H"]
        mu_model_strings = ["S - H", "1D - H", "Orig. 3D - H"]

        # difference between the SW heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            if difftype == 'diff_of_pure_amp':
                # m_s_diff = (m_s_sw - m_s_hom).abs()
                m_s_diff = m_s_sw - m_s_hom
            elif difftype == 'amp_of_diff_vector':
                m_s_diff = m_s_diff_swhom
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[0]}"])
            maxdiff = (m_s_diff[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.02
            maxdiff = 0.14
            diffstep = maxdiff/10
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="viridis", series=[0, maxdiff, diffstep], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            if sort:
                order = m_s_diff[col2plt].argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff[col2plt].iloc[order], cmap=True)
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_diff[col2plt], cmap=True)   # no smoothing or interpolation
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            # fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.text(text=f"@~D@~ locking: [{m_s_diff[col2plt].min():.2f}, {m_s_diff[col2plt].max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
                        
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")


        # difference between the original 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            if difftype == 'diff_of_pure_amp':
                # m_s_diff = (m_s_1d - m_s_hom).abs()
                m_s_diff = m_s_1d - m_s_hom
            elif difftype == 'amp_of_diff_vector':
                m_s_diff = m_s_diff_1dhom
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[1]}"])
            # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.016
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="viridis", series=[0, maxdiff, diffstep], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            if sort:            
                order = m_s_diff[col2plt].argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff[col2plt].iloc[order], cmap=True)
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_diff[col2plt], cmap=True)   # no smoothing or interpolation
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            # fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
            
            # fig.text(text=f"Min. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}", x=region[0]+0.1, y=region[-2]+0.2, font="6p,Helvetica,black", justify="ML")
            # fig.text(text=f"Max. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ locking: [{m_s_diff[col2plt].min():.2f}, {m_s_diff[col2plt].max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")

        # difference between the amplified 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            if difftype == 'diff_of_pure_amp':
                # m_s_diff = (m_s_3d_ori - m_s_hom).abs()
                m_s_diff = m_s_3d_ori - m_s_hom
            elif difftype == 'amp_of_diff_vector':
                m_s_diff = m_s_diff_3dohom 
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_strings[2]}"])
            # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # maxdiff = 0.16
            # diffstep = 0.016
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diffstep], background=True, reverse=False)
            # pygmt.makecpt(cmap="viridis", series=[0, maxdiff, diffstep], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            if sort:            
                order = m_s_diff[col2plt].argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff[col2plt].iloc[order], cmap=True)
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_diff[col2plt], cmap=True)   # no smoothing or interpolation
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            # fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
            
            # fig.text(text=f"Min. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}", x=region[0]+0.1, y=region[-2]+0.2, font="6p,Helvetica,black", justify="ML")
            # fig.text(text=f"Max. diff.: {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}", x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"@~D@~ locking: [{m_s_diff[col2plt].min():.2f}, {m_s_diff[col2plt].max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=[f"a{diffstep*5}f{diffstep}", "x+lLocking difference"], position="JBC+h+o0/0.5c")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_locking_diff_vector.pdf")

plot_slip_diff2(m_s_diff_swhom, m_s_diff_1dhom, m_s_diff_3dohom, 
                m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
                col2plt, region, difftype='diff_of_pure_amp', sort=True, flag_savefig=False)

plot_slip_diff2(m_s_diff_swhom, m_s_diff_1dhom, m_s_diff_3dohom, 
                m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori,
                col2plt, region, difftype='amp_of_diff_vector', sort=True, flag_savefig=False)


In [ ]:
def plot_inferred_locking_panel(fig, m_s, mom_rate, panel_idx, title_str, panel_label, region, col2plt, slipsybsz, colormap):
    
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    # fig.coast(region=regioqn, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
    #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
    maxslip = 1
    # maxslip = mtrue_s[col2plt].max()
    # maxslip = m_s_hom[col2plt].max()
    # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
    # slipstep = maxslip/10
    slipstep = 0.1
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)    #m_s_hom[col2plt].max()
    grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
    # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
    # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

    fig.text(text=f"Potency rate: {mom_rate[potency_col].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05, 
             y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Max. locking: {m_s[col2plt].max():.2f}", x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
 
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")  
     
    colorlabel = "Interseismic locking ratio"          
    with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

    #plot panel label
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    
    if panel_idx == 0:
        # Legend box bottom-left corner (adjust to your map's coordinates)
        x0, y0 = region[0]+0.05, region[-2]+0.35   # example, adjust manually
        # Box size
        width = 0.7    # degrees or projected units depending on your region
        height = 0.35
        # 1 — draw the gray box
        fig.plot(
            x=[x0, x0+width, x0+width, x0, x0],
            y=[y0, y0, y0+height, y0+height, y0],
            pen="0.5p,black",
            fill="lightgray",
            # transparency=30,
        )

        # Manual legend - more control and simpler
        # Position in your subplot coordinates
        x_legend = region[0] + 0.1  # adjust based on your region
        y_legend = region[2] + 0.45

        # Draw black line
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15],
                pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking",
                font="5p,Helvetica", justify="LM")

        # Draw white line
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend],
                pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking",
                font="5p,Helvetica", justify="LM")


def plot_inferred_locking_diff_panel(fig, m_s, mom_rate, title_str, panel_label, maxdiff, diff_step, 
                                  region, col2plt, slipsybsz, colormap):
    
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    # maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
    # maxdiff = 0.16
    # diffstep = 0.02
    # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
    pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
    # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
    # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
    # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
    # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
    # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
    # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
    # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
    # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

    m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
    order = m_s_diff.argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
    # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
    grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

    diff = mom_rate[potency_col].iloc[0] - mom_rate_hom[potency_col].iloc[0]
    fig.text(text=f"@~D@~ Potency rate: {diff:.2e} m@+3@+/yr",
             x=region[0]+0.05, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"@~D@~ locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
                x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
                
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
    # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
    grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
    fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

    colorlabel = "Difference in locking ratio"          
    with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
        # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
        fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

    #plot panel label
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")


def plot_all_inferred_locking_and_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d, 
                                       mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d, 
                                       col2plt, region, maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    colormap_diff = "roma"

    # mu_model_strings = ["H", "S", "3D", "S - H", "3D - H"]
    # mu_model_strings = ["H", "S", "Orig. 3D", "S - H", "Orig. 3D - H"]
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=2, ncols=4, figsize=("20c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        
        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_hom, mom_rate_hom, panel_idx, mu_model_strings[panel_idx], panel_labels[panel_idx], 
                                     region, col2plt, slipsybsz, colormap)

        # Slip inferred from the SW model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_sw, mom_rate_sw, panel_idx, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                    region, col2plt, slipsybsz, colormap)

        # Slip inferred from the 1D model
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_1d, mom_rate_1d, panel_idx, mu_model_strings[panel_idx], panel_labels[panel_idx], 
                                     region, col2plt, slipsybsz, colormap)
                                                 
        # Slip inferred from the 3D model
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx], panel_labels[panel_idx], 
                                     region, col2plt, slipsybsz, colormap)

        # difference between the SW heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_sw, mom_rate_sw, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

        # difference between the 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 2]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_1d, mom_rate_1d, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

        # difference between the 3D heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 3]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d, mom_rate_3d, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_locking_and_diff.pdf"
        figname = meshname + "_ref1D_locking_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxdiff = 0.16
    # diff_step = 0.02
    diff_step = maxdiff/10
else:
    maxdiff = 0.14    
    # diff_step = 0.02
    diff_step = maxdiff/10

plot_all_inferred_locking_and_diff(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, 
                                   mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori, 
                                   col2plt, region1, maxdiff, diff_step, flag_savefig=flag_savefig)


In [ ]:
# ── v2: shared colorbars in the empty [1,0] panel ──────────────────────────────
# Redefine helpers with optional show_cbar flag (default True = backward compatible)

def plot_inferred_locking_panel(fig, m_s, mom_rate, panel_idx, title_str, panel_label, region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    maxslip = 1
    slipstep = 0.1
    pygmt.makecpt(cmap=colormap, series=[0, maxslip, slipstep], background=True, reverse=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    order = m_s[col2plt].argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt], cmap=True)
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    fig.text(text=f"Max. locking: {m_s[col2plt].max():.2f}", x=region[0]+0.05, y=region[-2]+0.25,
             font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"Potency rate: {mom_rate[potency_col].iloc[0]:.2e} m@+3@+/yr", x=region[0]+0.05, 
             y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
    
    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=["af", "x+lInterseismic locking ratio"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
    if panel_idx == 0:
        x0, y0 = region[0]+0.05, region[-2]+0.35
        width = 0.7
        height = 0.35
        fig.plot(x=[x0, x0+width, x0+width, x0, x0], y=[y0, y0, y0+height, y0+height, y0],
                 pen="0.5p,black", fill="lightgray")
        x_legend = region[0] + 0.1
        y_legend = region[2] + 0.45
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend+0.15, y_legend+0.15], pen="1p,black")
        fig.text(x=x_legend+0.15, y=y_legend+0.15, text="90% locking", font="5p,Helvetica", justify="LM")
        fig.plot(x=[x_legend, x_legend+0.12], y=[y_legend, y_legend], pen="1p,white")
        fig.text(x=x_legend+0.15, y=y_legend, text="50% locking", font="5p,Helvetica", justify="LM")


def plot_inferred_locking_diff_panel(fig, m_s, mom_rate, title_str, panel_label, maxdiff, diff_step,
                                   region, col2plt, slipsybsz, colormap, show_cbar=True):
    fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str}"])
    pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
    m_s_diff = m_s[col2plt] - m_s_hom[col2plt]
    order = m_s_diff.argsort()
    fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
    # fig.plot(x=loc_f["lon"], y=loc_f["lat"], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)
    grid = pygmt.xyz2grd(x=loc_f["lon"], y=loc_f["lat"], z=m_s[col2plt], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
    fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
    delta_max = float(f"{m_s[col2plt].max():.2f}") - float(f"{m_s_hom[col2plt].max():.2f}")
    fig.text(text=f"@~D@~ Max. locking: {delta_max:+.2f}",
             x=region[0]+0.05, y=region[-2]+0.40, font="6p,Helvetica,black", justify="ML")
    fig.text(text=f"@~D@~ Locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
             x=region[0]+0.05, y=region[-2]+0.25, font="6p,Helvetica,black", justify="ML")
    diff = mom_rate[potency_col].iloc[0] - mom_rate_hom[potency_col].iloc[0]
    fig.text(text=f"@~D@~ Potency rate: {diff:.2e} m@+3@+/yr",
             x=region[0]+0.05, y=region[-2]+0.10, font="6p,Helvetica,black", justify="ML")

    scale_lon, scale_lat = region[1]-0.4, region[-1]-0.25
    mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
    fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
    grid2 = pygmt.xyz2grd(x=df_plate["lon"], y=df_plate["lat"], z=df_plate["z"], region=region, spacing=(0.05, 0.05))
    fig.grdcontour(grid=grid2, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray")
    fig.plot(x=trench["lon"], y=trench["lat"], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
    if show_cbar:
        with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
            fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lDifference in locking ratio"], position="JBC+h+o0/0.5c")
    fig.text(text=panel_label, x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")


def plot_all_inferred_locking_and_diff_v2(m_s_hom, m_s_sw, m_s_1d, m_s_3d, 
                                       mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d, 
                                       col2plt, region, maxdiff, diff_step, flag_savefig=False):
    slipsybsz = "c0.11c"
    colormap = "viridis"
    colormap_diff = "roma"
    mu_model_strings = ["H", "S", "1D", "Orig. 3D", "S - H", "1D - H", "Orig. 3D - H"]
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=4, figsize=("20c", "11.0c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.0c"], sharey=True):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_hom, mom_rate_hom, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_sw, mom_rate_sw, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 2]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_1d, mom_rate_1d, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap, show_cbar=False)
        with fig.set_panel(panel=[0, 3]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx],
                                     panel_labels[panel_idx], region, col2plt, slipsybsz, colormap, show_cbar=False)

        # ── Empty panel [1,0]: two shared colorbars ──────────────────────────
        with fig.set_panel(panel=[1, 0]):
            with pygmt.config(MAP_FRAME_PEN="0p,white", MAP_TICK_PEN_PRIMARY="0p,white"):
                fig.basemap(region=[0, 1, 0, 1], projection="X?/?", frame="0")
            # Colorbar 1: Interseismic locking ratio
            pygmt.makecpt(cmap=colormap, series=[0, 1, 0.1], background=True, reverse=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=["a0.2f0.1", "x+lInterseismic locking ratio"],
                             position="JMC+w3.8c/0.2c+h+o0c/2.4c")
            # Colorbar 2: Difference in locking ratio
            pygmt.makecpt(cmap=colormap_diff, series=[-maxdiff, maxdiff, diff_step], background=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", "x+lDifference in locking ratio"],
                             position="JMC+w3.8c/0.2c+h+o0c/1.2c")

        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_sw, mom_rate_sw, mu_model_strings[panel_idx],
                                          panel_labels[panel_idx], maxdiff, diff_step,
                                          region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 2]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_1d, mom_rate_1d, mu_model_strings[panel_idx],
                                          panel_labels[panel_idx], maxdiff, diff_step,
                                          region, col2plt, slipsybsz, colormap_diff, show_cbar=False)
        with fig.set_panel(panel=[1, 3]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d, mom_rate_3d, mu_model_strings[panel_idx],
                                          panel_labels[panel_idx], maxdiff, diff_step,
                                          region, col2plt, slipsybsz, colormap_diff, show_cbar=False)

    fig.show()
    if flag_savefig:
        figname = meshname + "_ref1D_locking_and_diff_v2.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)


plot_all_inferred_locking_and_diff_v2(m_s_hom, m_s_sw, m_s_1d, m_s_3d_ori, 
                                   mom_rate_hom, mom_rate_sw, mom_rate_1d, mom_rate_3d_ori, 
                                   col2plt, region1, maxdiff, diff_step, flag_savefig=flag_savefig)


In [ ]:
def plot_all_inferred_locking_and_diff_extra(m_s_hom, m_s_3d, mom_rate_hom, mom_rate_3d, 
                                          col2plt, region, maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    colormap_diff = "roma"

    mu_model_strings = ["Ampl. 3D", "Ampl. 3D - H"]
    # mu_model_strings = ["Ampl. 3D", "Ampl. 3D", "Ampl. 3D - H"]
    
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    # with fig.subplot(nrows=1, ncols=3, figsize=("15c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
    #                 margins=["0.1c", "0.25c"], sharey=True):

    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.25c"], sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0        
        # with fig.set_panel(panel=[0, 0]):
        #     plot_inferred_slip_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx],
        #                              panel_labels[panel_idx], region, 'mag', slipsybsz, colormap)

        # Slip inferred from the amplified 3D model
        with fig.set_panel(panel=[0, 0]):
            # panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx], 
                                        panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)

        # difference between the amplified 3D model and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d, mom_rate_3d, mu_model_strings[panel_idx], 
                                             panel_labels[panel_idx],maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_locking_and_diff.pdf"
        figname = meshname + "_ref1D_suppl_locking_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxdiff = 0.16
    # diff_step = 0.02
    diff_step = maxdiff/10
else:
    maxdiff = 0.14    
    # diff_step = 0.02
    diff_step = maxdiff/10

plot_all_inferred_locking_and_diff_extra(m_s_hom, m_s_3d, mom_rate_hom, mom_rate_3d, 
                                      col2plt, region1, maxdiff, diff_step, flag_savefig=flag_savefig)


In [ ]:
def plot_all_inferred_locking_and_diff_extra2(m_s_hom, m_s_1d, m_s_3d, mom_rate_hom, mom_rate_1d, mom_rate_3d, 
                                           col2plt, region, maxdiff, diff_step, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    colormap_diff = "roma"

    mu_model_strings = ["Ampl. 3D", "1D", "Ampl. 3D - H", "1D - H"]
    
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()

    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.5c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.25c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        panel_idx = 0

        # Slip inferred from the amplified 3D model
        with fig.set_panel(panel=[0, 0]):
            plot_inferred_locking_panel(fig, m_s_3d, mom_rate_3d, panel_idx, mu_model_strings[panel_idx], 
                                        panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)

        # Slip inferred from the 1D reference model
        with fig.set_panel(panel=[0, 1]):
            panel_idx += 1
            plot_inferred_locking_panel(fig, m_s_1d, mom_rate_1d, panel_idx, mu_model_strings[panel_idx], 
                                        panel_labels[panel_idx], region, col2plt, slipsybsz, colormap)

        # difference between the amplified 3D model and homogeneous model
        with fig.set_panel(panel=[1, 0]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_3d, mom_rate_3d, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

        # difference between the 1D reference model and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            panel_idx += 1
            plot_inferred_locking_diff_panel(fig, m_s_1d, mom_rate_1d, mu_model_strings[panel_idx], panel_labels[panel_idx],
                                          maxdiff, diff_step, region, col2plt, slipsybsz, colormap_diff)

    fig.show()

    if flag_savefig:
        # figname = meshname + "_locking_and_diff.pdf"
        figname = meshname + "_ref1D_suppl2_locking_and_diff.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

if not use_uneven_mesh:
    maxdiff = 0.16
    # diff_step = 0.02
    diff_step = maxdiff/10
else:
    maxdiff = 0.14    
    # diff_step = 0.02
    diff_step = maxdiff/10

plot_all_inferred_locking_and_diff_extra2(m_s_hom, m_s_1d, m_s_3d, mom_rate_hom, mom_rate_1d, mom_rate_3d,
                                       col2plt, region1, maxdiff, diff_step, flag_savefig=flag_savefig)


In [ ]:
def plot_homvshet_slip(m_s_hom, m_s, col2plt, region, struc_str_inv, flag_savefig=False):

    slipsybsz = "c0.09c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"]) #, "y+l(mm)"

        # slip inferred from the heterogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet. inv."])
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"])

        # difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["af", "+tHet - Hom"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="hot", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,white")
            fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")
            
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference"])

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_hethomslip.pdf")


plot_homvshet_slip(m_s_hom, m_s_sw, 'locking', region, sw_str, flag_savefig=False)    
plot_homvshet_slip(m_s_hom, m_s_3d, 'locking', region, het3d_str, flag_savefig=False)    
# plot_homvshet_slip(m_s_hom, m_s_3d_ori, 'locking', region, het3d_ori_str, flag_savefig=True)    


In [ ]:
# compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_obs)* 1000  # Convert to mm   
rmse_sw = utp.rmse_3d_vec_dataframe(u_pred_sw, u_obs)* 1000  # Convert to mm
rmse_1d = utp.rmse_3d_vec_dataframe(u_pred_1d, u_obs)* 1000  # Convert to mm
rmse_3d_ori = utp.rmse_3d_vec_dataframe(u_pred_3d_ori, u_obs)* 1000  # Convert to mm   
rmse_3d = utp.rmse_3d_vec_dataframe(u_pred_3d, u_obs)* 1000  # Convert to mm   
print(f"RMSE (mm): Hom={rmse_hom:.3f}, Het={rmse_sw:.3f}, 1D={rmse_1d:.3f},",
      f"3D-Ori={rmse_3d_ori:.3f}, 3D={rmse_3d:.3f}")

In [ ]:
# # plot the histogram of difference in slip
# fig, ax = plt.subplots(figsize=(4, 4.5), dpi=300)

# xmin, xmax = -0.25, 0.25
# bin_width = 0.01
# bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
# median = np.median(m_s_locking['diff'])
# sigma = np.std(m_s_locking['diff'])
# # rounded_sigma = round(sigma)
# counts1, _, _ = ax.hist(m_s_locking['diff'], bins=bins, color='dimgray', alpha=0.7, edgecolor='black')
# ymin, ymax = 0, np.max(counts1) + 100
# ax.errorbar(median, ymax*0.82, xerr=sigma, fmt='o', color='black', capsize=5, capthick=2, 
#             linewidth=2, label=fr'$\pm1\sigma = \pm{sigma:.3f}$')
# ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median = {median:.3f}')
# ax.legend()
# ax.set_xlabel('Difference')
# ax.set_ylabel('Frequency')
# ax.set_title('Difference in coupling (Het-Hom)', fontweight='bold')
# ax.set_xlim(xmin, xmax)
# ax.set_ylim(ymin, ymax)


In [ ]:
def build_pred_disp_vector(df, u_pred, u_res, error_e, error_n, error_v):
    ### data fitting by the heterogeneous inversion 
    # convert from m to mm
    df_pred_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_pred['ux']*1000,
            "north_velocity": u_pred['uy']*1000,        
        }
    )
    df_pred_h.head()

    df_pred_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_pred['uz']*1000,        
        }
    )

    # convert from m to mm, in order to directly compare with 
    df_res_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_res['ux']*1000,
            "north_velocity": u_res['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,        
        }
    )
    df_res_h.head()
    df_res_h_dum = df_res_h.iloc[:, :4]

    df_res_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_res['uz']*1000,   
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,     
        }
    )
    df_res_v_dum = df_res_v.iloc[:, :4]

    return df_pred_h, df_pred_v, df_res_h, df_res_v, df_res_h_dum, df_res_v_dum


# buil observation disp. vectors for various structure models
df_pred_h_hom, df_pred_v_hom, df_res_h_hom, df_res_v_hom, \
    df_res_h_dum_hom, df_res_v_dum_hom = build_pred_disp_vector(df, u_pred_hom, u_res_hom, error_e, error_n, error_v)

df_pred_h_sw, df_pred_v_sw, df_res_h_sw, df_res_v_sw, \
    df_res_h_dum_sw, df_res_v_dum_sw = build_pred_disp_vector(df, u_pred_sw, u_res_sw, error_e, error_n, error_v)

df_pred_h_1d, df_pred_v_1d, df_res_h_1d, df_res_v_1d, \
    df_res_h_dum_1d, df_res_v_dum_1d = build_pred_disp_vector(df, u_pred_1d, u_res_1d, error_e, error_n, error_v)

df_pred_h_3d, df_pred_v_3d, df_res_h_3d, df_res_v_3d, \
    df_res_h_dum_3d, df_res_v_dum_3d = build_pred_disp_vector(df, u_pred_3d, u_res_3d, error_e, error_n, error_v)

df_pred_h_3d_ori, df_pred_v_3d_ori, df_res_h_3d_ori, df_res_v_3d_ori, \
    df_res_h_dum_3d_ori, df_res_v_dum_3d_ori = build_pred_disp_vector(df, u_pred_3d_ori, u_res_3d_ori, error_e, error_n, error_v)


In [ ]:
def plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h, df_res_h, df_res_h_dum, 
                               u_true, u_pred, color, lgdstr, 
                               region, struc_str_inv, errorbar=True, flag_savefig=False):
    # plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_h, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.05/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.5/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_h, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            else:
                fig.velo(data=df_res_h_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a60+p1p,{color}+ea+g{color}")       
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse = utp.rmse_3d_vec_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                    "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")

    fig.show()
    
    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_datafit_h.pdf")


plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h_hom, df_res_h_hom, df_res_h_dum_hom, 
                           u_obs, u_pred_hom, 'blue', 'hom', region1, homo_str, errorbar=True, flag_savefig=True)
plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h_sw, df_res_h_sw, df_res_h_dum_sw, 
                           u_obs, u_pred_sw, 'red', 'het', region1, sw_str, errorbar=True, flag_savefig=False)
plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h_3d, df_res_h_3d, df_res_h_dum_3d, 
                           u_obs, u_pred_3d, 'red', 'het', region1, het3d_str, errorbar=True, flag_savefig=False)
plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h_3d_ori, df_res_h_3d_ori, df_res_h_dum_3d_ori, 
                           u_obs, u_pred_3d_ori, 'red', 'het', region1, het3d_ori_str, errorbar=True, flag_savefig=False)


In [ ]:
def plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v, df_res_v, df_res_v_dum, u_true, u_pred, 
                                color, lgdstr, region, struc_str_inv, errorbar=True, flag_savefig=False):
    # plot the vertical predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_v, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.05/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.5/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_v, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            else:
                fig.velo(data=df_res_v_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse = utp.rmse_3d_vec_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_datafit_v.pdf")


plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v_hom, df_res_v_hom, df_res_v_dum_hom, 
                            u_obs, u_pred_hom, 'blue', 'hom', region1, homo_str, errorbar=True, flag_savefig=True)
plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v_sw, df_res_v_sw, df_res_v_dum_sw, 
                            u_obs, u_pred_sw, 'red', 'het', region1, sw_str, errorbar=True, flag_savefig=False)
plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v_3d, df_res_v_3d, df_res_v_dum_3d, 
                            u_obs, u_pred_3d, 'red', 'het', region1, het3d_str, errorbar=True, flag_savefig=False)


In [ ]:
def plot_disp_obs_pred_res(df_obs_h, df_pred_hom_h, df_pred_h, df_res_hom_h_dum, df_res_h_dum, 
                           df_obs_v, df_pred_hom_v, df_pred_v, df_res_hom_v_dum, df_res_v_dum, 
                           u_true, u_pred_hom, u_pred,  
                           region, struc_str_inv, flag_savefig=False):
    # plot the predicted displacements vs. observed at GNSS stations, and the residual for ALL components
    
    ### Plot the comparison on horizontal components
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=3, figsize=("18c", "12c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # homogeneous Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")        

        # heterogeneous Observed vs. predicted
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # residuals, hom vs het
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_hom_h_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_h_dum, pen="0.5p,red", line="0.25p,red", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,red+ea+gred")

            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm   
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse = utp.rmse_3d_vec_dataframe(u_pred, u_true)* 1000  # Convert to mm  
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2, font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4, font="9p,Helvetica,red", justify="ML")

            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        ### Plot the comparison on vectical components
        # homogeneous Observed vs. predicted
        with fig.set_panel(panel=[1, 0]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_hom_v, pen="0.5p,blue", line="0.25p,blue", spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")      

        # heterogeneous Observed vs. predicted
        with fig.set_panel(panel=[1, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_v, pen="0.5p,red", line="0.25p,red", spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # residuals, hom vs het
        with fig.set_panel(panel=[1, 2]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_hom_v_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_v_dum, pen="0.5p,red", line="0.25p,red", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,red+ea+gred")

            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm   
            # compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
            rmse = utp.rmse_3d_vec_dataframe(u_pred, u_true)* 1000  # Convert to mm  
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2, font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4, font="9p,Helvetica,red", justify="ML")

            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_datafit.pdf")


plot_disp_obs_pred_res(df_obs_h, df_pred_h_hom, df_pred_h_sw, df_res_h_dum_hom, df_res_h_dum_sw, 
                       df_obs_v, df_pred_v_hom, df_pred_v_sw, df_res_v_dum_hom, df_res_v_dum_sw, 
                       u_obs, u_pred_hom, u_pred_sw, 
                       region1, sw_str, flag_savefig=True)

plot_disp_obs_pred_res(df_obs_h, df_pred_h_hom, df_pred_h_3d, df_res_h_dum_hom, df_res_h_dum_3d, 
                       df_obs_v, df_pred_v_hom, df_pred_v_3d, df_res_v_dum_hom, df_res_v_dum_3d, 
                       u_obs, u_pred_hom, u_pred_3d,
                       region1, het3d_str, flag_savefig=True)

plot_disp_obs_pred_res(df_obs_h, df_pred_h_hom, df_pred_h_3d_ori, df_res_h_dum_hom, df_res_h_dum_3d_ori, 
                       df_obs_v, df_pred_v_hom, df_pred_v_3d_ori, df_res_v_dum_hom, df_res_v_dum_3d_ori, 
                       u_obs, u_pred_hom, u_pred_3d_ori,
                       region1, het3d_ori_str, flag_savefig=True)

In [ ]:
# Rose diagram: distribution of horizontal residual vector azimuths (free-slip inversion)
# Residual = obs - pred. Reference lines: plate convergence azimuth and its
# anti-parallel (red solid, one combined legend entry); per-panel weighted
# median residual direction (black dashed). Median is preferred over mean
# because R is moderate — the residual distributions are skewed and the mean
# is pulled away from the visual mode of the rose.

plate_az = 33.5   # plate convergence azimuth CW from North (reference only)

bin_deg = 5
bins        = np.arange(0, 361, bin_deg)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
theta = np.deg2rad(bin_centers)
width = np.deg2rad(bin_deg)

labels  = ["H", "S", "1D", "Orig. 3D", "Ampl. 3D"]
res_dfs = [df_res_h_hom, df_res_h_sw, df_res_h_1d, df_res_h_3d_ori, df_res_h_3d]

# Choose how to weight each residual:
#   'equal' → every station counts as 1 (pure directional distribution)
#   'amp'   → weight by |res| (mm/yr) so larger residuals dominate
weight_mode = 'amp'        # 'equal' or 'amp'

# Pre-compute weighted counts, RMSE, weighted-mean direction (and median + R for
# diagnostic printing) so we can set a shared radial limit across panels.
all_counts = []
all_az     = []
all_amp    = []
all_rmse   = []
all_mean   = []
all_median = []
all_R      = []
for df_res in res_dfs:
    e, n = df_res['east_velocity'].values, df_res['north_velocity'].values
    az_deg = np.degrees(np.arctan2(e, n)) % 360
    amp    = np.hypot(e, n)
    rmse   = np.sqrt(np.mean(amp**2))

    if weight_mode == 'equal':
        weights = np.ones_like(amp)
    elif weight_mode == 'amp':
        weights = amp
    else:
        raise ValueError(f"unknown weight_mode: {weight_mode!r}")

    sin_s = np.sum(weights * np.sin(np.deg2rad(az_deg)))
    cos_s = np.sum(weights * np.cos(np.deg2rad(az_deg)))
    az_mean = np.degrees(np.arctan2(sin_s, cos_s)) % 360
    R       = np.hypot(sin_s, cos_s) / np.sum(weights)

    az_centered = ((az_deg - az_mean + 180) % 360) - 180
    order_c     = np.argsort(az_centered)
    cum_w       = np.cumsum(weights[order_c]) / np.sum(weights)
    med_idx     = np.searchsorted(cum_w, 0.5)
    az_median   = (az_centered[order_c][med_idx] + az_mean) % 360

    counts, _ = np.histogram(az_deg, bins=bins, weights=weights)
    all_counts.append(counts)
    all_az.append(az_deg)
    all_amp.append(amp)
    all_rmse.append(rmse)
    all_mean.append(az_mean)
    all_median.append(az_median)
    all_R.append(R)

rmax = np.ceil(max(c.max() for c in all_counts) * 1.05)  # 5% headroom
rmax = 27   # use the same as in az version for better comparison
print(f"rmax: {rmax:.1f}")

fig_rose, axes = plt.subplots(2, 4, figsize=(12, 8), dpi=300,
                               subplot_kw=dict(projection='polar'))
axes = axes.ravel()
for i, (ax, counts, az_deg, rmse, az_median, label) in enumerate(zip(
        axes[:5], all_counts, all_az, all_rmse, all_median, labels)):
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)   # clockwise = geographic convention
    ax.bar(theta, counts, width=width, bottom=0,
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_ylim(0, rmax)
    # Move the radial tick labels along NW, clear of the rose bars in the NE quadrant.
    ax.set_rlabel_position(315)

    # Reference lines: plate convergence azimuth + anti-parallel (red solid, one
    # combined legend entry on panel 0) + per-panel weighted median (black dashed).
    h_pres, = ax.plot([], [], color='red', lw=1.5, linestyle='-',
                       label=f"plate az. {plate_az:.1f}° / {(plate_az + 180) % 360:.1f}°")
    h_med,  = ax.plot([], [], color='k', lw=1.8, linestyle='--',
                       label="weighted median")
    ax.axvline(np.deg2rad(plate_az % 360),         color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad((plate_az + 180) % 360), color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad(az_median), color='k', lw=1.8, linestyle='--')

    # Annotate the median angle just CCW (toward N) of its line, mid-radius —
    # away from the rose bars in the NE quadrant.
    ax.text(np.deg2rad(az_median + 15),
            rmax * 0.8,
            f"{az_median:.1f}°",
            ha='center', va='center',
            fontsize=9, color='k')

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax.set_title(f"{label}  (RMSE$_h$={rmse:.2f} mm/yr)", va='bottom', pad=25, fontsize=10)

    if i == 0:
        ax.legend(handles=[h_pres, h_med],
                  loc='lower left', bbox_to_anchor=(-0.15, -0.30),
                  fontsize=9, frameon=False)

# Hide all trailing unused panels
for ax in axes[len(labels):]:
    ax.axis('off')

plt.suptitle(f"Horizontal residual azimuths  (obs − pred)  "
             f"(n={len(all_az[0])}, bin={bin_deg}°)",
             y=0.97, fontsize=11)
plt.tight_layout()
# if flag_savefig:
#     fig_rose.savefig(figpath + meshname + f"_residual_az_rose_{weight_mode}.pdf",
#                      bbox_inches='tight')
plt.show()

# Diagnostic print: per-model R + weighted mean & median
print(f"weight_mode = {weight_mode!r}")
for lbl, az_m, az_md, R in zip(labels, all_mean, all_median, all_R):
    print(f"  {lbl:>10s}: mean={az_m:6.2f}°  median={az_md:6.2f}°  R={R:.3f}")
print(f"Plate convergence azimuth: {plate_az:.2f}° / anti-parallel {(plate_az + 180) % 360:.2f}°")


In [ ]:
# Production version of the residual rose: only the 4 main μ models
# (H, S, 1D, Orig. 3D) in a single row.  Re-uses the per-model statistics
# already computed in the cell above (all_counts, all_az, all_rmse, all_median,
# labels), so this cell must run after the 5-panel reference cell.
# Ampl. 3D goes into the 5-panel reference above (kept for the supplement).

n_main  = 4
labels_m = labels[:n_main]

# Recompute rmax from all_counts so this cell is idempotent — independent
# of any cells that may have shadowed the module-level `rmax`.
rmax = np.ceil(max(c.max() for c in all_counts) * 1.05)  # 5% headroom
rmax = 27   # use the same as in az version for better comparison
print(f"rmax: {rmax:.1f}")

fig_rose, axes = plt.subplots(1, n_main, figsize=(12, 3.6), dpi=300,
                               subplot_kw=dict(projection='polar'))
for i, (ax, counts, az_deg, rmse, az_median, label) in enumerate(zip(
        axes, all_counts[:n_main], all_az[:n_main],
        all_rmse[:n_main], all_median[:n_main], labels_m)):
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.bar(theta, counts, width=width, bottom=0,
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_ylim(0, rmax)
    ax.set_rlabel_position(315)

    h_pres, = ax.plot([], [], color='red', lw=1.5, linestyle='-',
                       label=f"V$_{{plate}}$ azimuth {plate_az:.1f}° / {(plate_az + 180) % 360:.1f}°")
    h_med,  = ax.plot([], [], color='k', lw=1.8, linestyle='--',
                       label="weighted median")
    ax.axvline(np.deg2rad(plate_az % 360),         color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad((plate_az + 180) % 360), color='red', lw=1.5, linestyle='-')
    ax.axvline(np.deg2rad(az_median), color='k', lw=1.8, linestyle='--')

    ax.text(np.deg2rad(az_median + 15), rmax * 0.8,
            f"{az_median:.1f}°",
            ha='center', va='center',
            fontsize=9, color='k')

    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax.set_title(f"{label}  (RMSE$_h$={rmse:.2f} mm/yr)", va='bottom', pad=25, fontsize=10)

    if i == 0:
        ax.legend(handles=[h_pres, h_med],
                  loc='lower left', bbox_to_anchor=(-0.10, -0.35),
                  fontsize=9, frameon=False)

plt.suptitle(f"Two-component inversion: Horizontal residual azimuths (obs − pred) "
             f"(n={len(all_az[0])}, bin={bin_deg}°)",
             y=1.02, fontsize=11)
plt.tight_layout()
if flag_savefig:
    fig_rose.savefig(figpath + meshname + "_residual_az_rose.pdf",
                     bbox_inches='tight')
    print(f"Saved: {figpath + meshname + '_residual_az_rose.pdf'}")
plt.show()


In [ ]:
# def plot_res_hist(u_true, u_pred_hom, u_res_hom, u_pred, u_res, struc_str_inv, flag_savefig=False):

#     # plot the histogram of difference in residuals
#     fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.5), dpi=300)

#     bin_width = 0.25

#     ax = axes[0]

#     xmin, xmax = 0, 5 
#     bins = np.arange(xmin, xmax + bin_width, bin_width)
#     counts1, _, _ = ax.hist(u_res_hom['mag']*1000, bins=bins, color='blue', alpha=0.7, edgecolor='black',label='Homogeneous')
#     counts2, _, _ = ax.hist(u_res['mag']*1000, bins=bins, color='red', alpha=0.7, edgecolor='black',label='Heterogeneous')

#     ymin, ymax = 0, max(np.max(counts1), np.max(counts2)) + 4

#     median_hom = np.median(u_res_hom['mag']*1000)
#     sigma_hom = np.std(u_res_hom['mag']*1000)
#     median = np.median(u_res['mag']*1000)
#     sigma = np.std(u_res['mag']*1000)

#     rmse_hom = utp.rmse_3d_vec_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm
#     print(f"HOM Residual RMSE: {rmse_hom:.2f} mm")
#     print(f"HOM Residual median: {median_hom:.2f} mm, sigma: {sigma_hom:.2f} mm")

#     rmse = utp.rmse_3d_vec_dataframe(u_pred, u_true)* 1000  # Convert to mm
#     print(f"HET Residual RMSE: {rmse:.2f} mm")
#     print(f"HET Residual median: {median:.2f} mm, sigma: {sigma:.2f} mm")

#     ax.plot([median_hom, median_hom], [ymin, ymax], '--', color='blue', label=fr'Hom median={median_hom:.1f}')
#     ax.plot([median, median], [ymin, ymax], '--', color='red', label=fr'Het median={median:.1f}')
#     ax.errorbar(median_hom, ymax-2, xerr=sigma_hom, fmt='o', color='blue', capsize=5, capthick=2, 
#                 linewidth=2, label=fr'Hom $\pm1\sigma=\pm{sigma_hom:.1f}$')
#     ax.errorbar(median, ymax-2, xerr=sigma, fmt='o', color='red', capsize=5, capthick=2, 
#                 linewidth=2, label=fr'Het $\pm1\sigma=\pm{sigma:.1f}$')
#     ax.text(0.5, 0.9, f"Hom Res RMSE:{rmse_hom:.2f}", fontsize=9, color='k', transform=ax.transAxes)
#     ax.text(0.5, 0.8, f"Het Res RMSE:{rmse:.2f}", fontsize=9, color='k', transform=ax.transAxes)

#     ax.legend(fontsize=9, loc='right')
#     ax.set_xlabel('Residual (mm)')
#     ax.set_ylabel('Frequency')
#     ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
#     ax.set_ylim(ymin, ymax)
#     ax.set_yticks(np.arange(ymin, ymax+2, 2))
#     ax.set_xlim(xmin, xmax)

#     ax = axes[1]
#     xmin, xmax = -2, 2
#     bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
#     counts1, _, _ = ax.hist(u_res['mag']*1000-u_res_hom['mag']*1000, bins=bins, color='dimgray',alpha=0.7, edgecolor='black',label='het-hom')
#     ymin, ymax = 0, np.max(counts1) + 2

#     median = np.median(u_res['mag']*1000-u_res_hom['mag']*1000)
#     ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median={median:.2f}')

#     ax.legend(fontsize=9, loc='right')
#     ax.set_xlabel('Difference (mm)')
#     ax.set_ylabel('Frequency')
#     ax.set_title('Residual difference (Het-Hom)', fontweight='bold')
#     ax.set_ylim(ymin, ymax)
#     ax.set_yticks(np.arange(ymin, ymax+2, 2))
#     ax.set_xlim(xmin, xmax)

#     if flag_savefig:
#         fig.savefig(figpath + meshname + struc_str_inv + "_res.pdf")


# plot_res_hist(u_obs, u_pred_hom, u_res_hom, u_pred_sw, u_res_sw, sw_str, flag_savefig=True)        
# plot_res_hist(u_obs, u_pred_hom, u_res_hom, u_pred_3d, u_res_3d, het3d_str, flag_savefig=True)              
